# Predição dos modelos V4.1, V5 e V5,1 nas Imagens dos Lotes 4 e 5

Autor:  Viviane da Rosa Sommer

Atualizado: 07/04/2025

## Objetivo: Comparar os resultados dos modelos V4.1, V5 e V5,1 com as imagens segmentadas dos especialistas, que pertencem aos Lotes 4 e 5.


## Importações Necessárias

In [ ]:
'''!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas'''

import glob
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import traceback 
import torchvision
import torch
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import auc 
from typing import List, Dict, Any, Tuple, Optional

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
mso_v1 = YOLO('Models/coral_yolov11_segm_2k_V4.1.pt')
mso_v2 = YOLO('Models/coral_yolov11_segm_4c_V5.pt')
mso_v3 = YOLO('Models/coral_yolov11_segm_2k_V5.1.pt')

INPUT_DIRECTORY_POSITIVE_IMAGES = "Datasets/Lote-4-5-Incompleto/original_images"
all_images = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.[pPjJ][nNpP][gGgG]") 

## Funções Necessárias

In [ ]:
def read_image_with_annotation(image_path: str, mask_path: str) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Reads the original image and its corresponding mask. Returns None for the mask if no contours are found.

    Args:
        image_path (str): Path to the original image.
        mask_path (str): Path to the mask image.

    Returns:
        Tuple[np.ndarray, Optional[np.ndarray]]: Original image and binary mask, or None if the mask is negative.

    Raises:
        ValueError: If the image or mask cannot be loaded.
    """
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Unable to load image: {image_path}")

    mask = cv2.imread(mask_path)
    if mask is None:
        raise ValueError(f"Unable to load mask: {mask_path}")

    green_lower_bound = np.array([0, 100, 0])
    green_upper_bound = np.array([100, 255, 100])
    binary_mask = cv2.inRange(mask, green_lower_bound, green_upper_bound).astype(np.uint8)

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return image, None

    return image, binary_mask
    
def process_results(results: List[Any]) -> Optional[Any]:
    """
    Extracts the first valid result containing masks from the model's output.

    Args:
        results (List[Any]): List of model detection results.

    Returns:
        Optional[Any]: The first valid result with masks, or None if no valid result exists.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.ndarray, model: Any) -> np.ndarray:
    """
    Predicts coral regions in an image using the specified model.

    Args:
        img (np.ndarray): Input image.
        model (Any): Model to use for prediction.

    Returns:
        np.ndarray: Binary mask for coral regions.
    """
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    if coral_best_result and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_masks) > 0:
            nms_indices = torchvision.ops.nms(boxes[coral_indices, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int().cpu().numpy() * 255
        else:
            final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    else:
        final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)

    return cv2.resize(final_coral_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)

def generate_predictions(files: List[str], model_v1: Any, model_v2: Any, model_v3: Any) -> Tuple[List[Dict[str, Any]], List[int]]:
    """
    Generates predictions for a list of images using three models and assigns labels based on the presence of contours.

    Args:
        files (List[str]): List of image paths.
        model_v1 (Any): First model for prediction.
        model_v2 (Any): Second model for prediction.
        model_v3 (Any): Third model for prediction.

    Returns:
        Tuple[List[Dict[str, Any]], List[int]]: List of dictionaries containing results for each file and the generated labels.
    """
    results = []
    labels = []  # List to store dynamic labels (1 for positive, 0 for negative)

    for image_path in files:
        try:
            mask_path = 'Datasets/Lote-4-5-Incompleto/label_images/label_' + image_path.split("/")[-1]
            cropped_image, annotation_mask = read_image_with_annotation(image_path, mask_path)

            if annotation_mask is None or annotation_mask.sum() == 0:
                labels.append(0)  # Negative label
                results.append({
                    "file": image_path,
                    "cropped_image": cropped_image,
                    "annotation_mask": None,
                    "mask_v1": None,
                    "mask_v2": None,
                    "mask_v3": None,
                    "iou_v1": float('nan'),
                    "iou_v2": float('nan'),
                    "iou_v3": float('nan'),
                    "dice_v1": float('nan'),
                    "dice_v2": float('nan'),
                    "dice_v3": float('nan')
                })
                continue

            labels.append(1)  # Positive label

            mask_v1_result = prediction_coral(cropped_image, model_v1)
            mask_v2_result = prediction_coral(cropped_image, model_v2)
            mask_v3_result = prediction_coral(cropped_image, model_v3)

            iou_v1 = calculate_iou(mask_v1_result, annotation_mask)
            iou_v2 = calculate_iou(mask_v2_result, annotation_mask)
            iou_v3 = calculate_iou(mask_v3_result, annotation_mask)

            dice_v1 = calculate_dice(mask_v1_result, annotation_mask)
            dice_v2 = calculate_dice(mask_v2_result, annotation_mask)
            dice_v3 = calculate_dice(mask_v3_result, annotation_mask)

            results.append({
                "file": image_path,
                "cropped_image": cropped_image,
                "annotation_mask": annotation_mask,
                "mask_v1": mask_v1_result,
                "mask_v2": mask_v2_result,
                "mask_v3": mask_v3_result,
                "iou_v1": iou_v1,
                "iou_v2": iou_v2,
                "iou_v3": iou_v3,
                "dice_v1": dice_v1,
                "dice_v2": dice_v2,
                "dice_v3": dice_v3
            })

        except Exception as e:
            print(f"Error processing {image_path} and {mask_path}: {e}")
            traceback.print_exc()

    return results, labels

def load_annotations_with_holes(annotation_path: str, image_size: Tuple[int, int]) -> np.ndarray:
    """
    Loads annotations from a YOLO segmentation file, handling holes (inner contours).

    Args:
        annotation_path (str): Path to the YOLO annotation file.
        image_size (Tuple[int, int]): Size of the image (height, width).

    Returns:
        np.ndarray: Binary mask where annotated regions (including holes) are marked with 255.
    """
    try:
        mask = np.zeros(image_size, dtype=np.uint8)

        with open(annotation_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                _x_center, _y_center, _width, _height = map(float, parts[1:5])
                points = np.array(list(map(float, parts[5:])))
                points = points.reshape(-1, 2)
                points[:, 0] *= image_size[1]
                points[:, 1] *= image_size[0]
                points = points.astype(np.int32)
                cv2.fillPoly(mask, [points], 255)

        return mask

    except Exception as e:
        print(f"Error reading annotation file {annotation_path}: {e}")
        traceback.print_exc()
        return np.zeros(image_size, dtype=np.uint8)

def draw_contours(ax: Any, image: np.ndarray, mask: Optional[np.ndarray], title: str, color: Tuple[int, int, int]) -> None:
    """
    Draws contours of a mask on top of an image.

    Args:
        ax (Any): Matplotlib axis for plotting.
        image (np.ndarray): Original image.
        mask (Optional[np.ndarray]): Binary mask for contours. Can be None.
        title (str): Title of the plot.
        color (Tuple[int, int, int]): Color of the contours.

    Returns:
        None
    """
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(title, fontsize=14)

    if mask is None or mask.sum() == 0:
        ax.text(0.5, 0.5, "No contours", fontsize=14, color='red', ha='center', va='center', transform=ax.transAxes)
        return

    contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        cv2.drawContours(image, [contour], -1, color, 2)

    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

def plot_image_comparisons(results: List[Dict[str, Any]]) -> None:
    """
    Plots comparisons of original images with annotation contours and predictions from three models.

    Args:
        results (List[Dict[str, Any]]): List of results containing masks and predictions.

    Returns:
        None
    """
    for result in results:
        try:
            cropped_image = result["cropped_image"]
            annotation_mask = result["annotation_mask"]
            mask_v1 = result["mask_v1"]
            mask_v2 = result["mask_v2"]
            mask_v3 = result["mask_v3"]
            image_name = result["file"].split('/')[-1]

            fig, axes = plt.subplots(2, 2, figsize=(20, 20))
            ax1, ax2, ax3, ax4 = axes.flatten()

            draw_contours(ax1, cropped_image.copy(), annotation_mask, 'Annotation Contour', (0, 255, 0))
            draw_contours(ax2, cropped_image.copy(), mask_v1, 'Prediction Model v1', (255, 0, 0))
            draw_contours(ax3, cropped_image.copy(), mask_v2, 'Prediction Model v2', (0, 0, 255))
            draw_contours(ax4, cropped_image.copy(), mask_v3, 'Prediction Model v3', (255, 255, 0))

            fig.suptitle(f"Image: {image_name}", fontsize=16)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"Error plotting {result['file']}: {e}")
            traceback.print_exc()

def evaluate_predictions(results: List[Dict[str, Any]], labels: List[int]) -> None:
    """
    Evaluates model predictions (v1, v2, v3) by comparing them with ground-truth labels.

    Args:
        results (List[Dict[str, Any]]): List of dictionaries containing masks and predictions.
        labels (List[int]): List of ground-truth labels (1 for positive, 0 for negative).

    Returns:
        None
    """
    predictions_v1 = [1 if result["mask_v1"] is not None and result["mask_v1"].sum() > 0 else 0 for result in results]
    predictions_v2 = [1 if result["mask_v2"] is not None and result["mask_v2"].sum() > 0 else 0 for result in results]
    predictions_v3 = [1 if result["mask_v3"] is not None and result["mask_v3"].sum() > 0 else 0 for result in results]

    all_classes = [0, 1]
    target_names = ["Negative", "Positive"]

    conf_matrix_v1 = confusion_matrix(labels, predictions_v1, labels=all_classes)
    conf_matrix_v2 = confusion_matrix(labels, predictions_v2, labels=all_classes)
    conf_matrix_v3 = confusion_matrix(labels, predictions_v3, labels=all_classes)

    print("Classification Report for Model v1:")
    print(classification_report(labels, predictions_v1, target_names=target_names, labels=all_classes, zero_division=0))

    print("\nClassification Report for Model v2:")
    print(classification_report(labels, predictions_v2, target_names=target_names, labels=all_classes, zero_division=0))

    print("\nClassification Report for Model v3:")
    print(classification_report(labels, predictions_v3, target_names=target_names, labels=all_classes, zero_division=0))

    plot_confusion_matrices_side_by_side(
        conf_matrices=[conf_matrix_v1, conf_matrix_v2, conf_matrix_v3],
        model_names=["Model v1", "Model v2", "Model v3"],
        class_names=target_names
    )
    

def plot_confusion_matrices_side_by_side(conf_matrices: List[np.ndarray], model_names: List[str], class_names: List[str]) -> None:
    """
    Plots multiple confusion matrices side by side.

    Args:
        conf_matrices (List[np.ndarray]): List of confusion matrices.
        model_names (List[str]): List of model names corresponding to each confusion matrix.
        class_names (List[str]): List of class names (e.g., ["Negative", "Positive"]).

    Returns:
        None
    """
    num_matrices = len(conf_matrices)
    plt.figure(figsize=(6 * num_matrices, 6))

    for i, (conf_matrix, model_name) in enumerate(zip(conf_matrices, model_names)):
        plt.subplot(1, num_matrices, i + 1)
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")

    plt.tight_layout()
    plt.show()

def calculate_iou(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float:
    """
    Calculates the Intersection over Union (IoU) between a predicted mask and annotation mask.

    Args:
        predicted_mask (np.ndarray): Predicted binary mask.
        annotation_mask (np.ndarray): Ground-truth binary mask.

    Returns:
        float: IoU value.
    """
    if predicted_mask is None or annotation_mask is None:
        return float('nan')

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    if predicted_mask.sum() == 0 and annotation_mask.sum() == 0:
        return 1.0

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()
    union = np.logical_or(predicted_mask, annotation_mask).sum()

    return intersection / union if union > 0 else 1.0


def calculate_dice(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float:
    """
    Calculates the Dice coefficient between a predicted mask and annotation mask.

    Args:
        predicted_mask (np.ndarray): Predicted binary mask.
        annotation_mask (np.ndarray): Ground-truth binary mask.

    Returns:
        float: Dice coefficient.
    """
    if predicted_mask is None or annotation_mask is None:
        return float('nan')

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    if predicted_mask.sum() == 0 and annotation_mask.sum() == 0:
        return 1.0

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()
    predicted_sum = predicted_mask.sum()
    annotation_sum = annotation_mask.sum()

    return (2 * intersection) / (predicted_sum + annotation_sum) if (predicted_sum + annotation_sum) > 0 else 0.0

def display_statistics(results: List[Dict[str, Any]]) -> None:
    """
    Displays statistics (mean, median, etc.) for IoU and Dice predictions for all three models.

    Args:
        results (List[Dict[str, Any]]): List of dictionaries containing IoU and Dice values.

    Returns:
        None
    """
    iou_v1_values = [result["iou_v1"] for result in results if not np.isnan(result["iou_v1"])]
    iou_v2_values = [result["iou_v2"] for result in results if not np.isnan(result["iou_v2"])]
    iou_v3_values = [result["iou_v3"] for result in results if not np.isnan(result["iou_v3"])]

    dice_v1_values = [result["dice_v1"] for result in results if not np.isnan(result["dice_v1"])]
    dice_v2_values = [result["dice_v2"] for result in results if not np.isnan(result["dice_v2"])]
    dice_v3_values = [result["dice_v3"] for result in results if not np.isnan(result["dice_v3"])]

    print("IoU Statistics for Model v1:")
    print(f"Mean IoU: {np.mean(iou_v1_values):.4f}")
    print(f"Median IoU: {np.median(iou_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v1_values):.4f}")

    print("\nIoU Statistics for Model v2:")
    print(f"Mean IoU: {np.mean(iou_v2_values):.4f}")
    print(f"Median IoU: {np.median(iou_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v2_values):.4f}")

    print("\nIoU Statistics for Model v3:")
    print(f"Mean IoU: {np.mean(iou_v3_values):.4f}")
    print(f"Median IoU: {np.median(iou_v3_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v3_values):.4f}")

    print("\nDice Statistics for Model v1:")
    print(f"Mean Dice: {np.mean(dice_v1_values):.4f}")
    print(f"Median Dice: {np.median(dice_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v1_values):.4f}")

    print("\nDice Statistics for Model v2:")
    print(f"Mean Dice: {np.mean(dice_v2_values):.4f}")
    print(f"Median Dice: {np.median(dice_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v2_values):.4f}")

    print("\nDice Statistics for Model v3:")
    print(f"Mean Dice: {np.mean(dice_v3_values):.4f}")
    print(f"Median Dice: {np.median(dice_v3_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v3_values):.4f}")

def calculate_map(results: List[Dict[str, Any]], iou_thresholds: List[float] = [0.5]) -> Dict[str, Dict[str, float]]:
    """
    Computes the Mean Average Precision (mAP) for model predictions at specified IoU thresholds.

    Args:
        results (List[Dict[str, Any]]): List of dictionaries containing predictions and annotations.
        iou_thresholds (List[float]): List of IoU thresholds for mAP calculation.

    Returns:
        Dict[str, Dict[str, float]]: Dictionary with mAP values for each model at the specified thresholds.
    """
    def compute_precision_recall(predictions: List[Optional[np.ndarray]], 
                                 annotations: List[Optional[np.ndarray]], 
                                 scores: List[float], 
                                 iou_threshold: float) -> Tuple[List[float], List[float]]:
        precisions = []
        recalls = []
        total_positives = sum(1 for mask in annotations if mask is not None and mask.sum() > 0)

        for threshold in np.linspace(0, 1, num=101):
            true_positives = 0
            false_positives = 0

            for pred_mask, annotation_mask, score in zip(predictions, annotations, scores):
                if annotation_mask is None or annotation_mask.sum() == 0:
                    if pred_mask is not None and pred_mask.sum() > 0 and score >= threshold:
                        false_positives += 1
                    continue

                if pred_mask is not None:
                    iou = calculate_iou(pred_mask, annotation_mask)
                    if iou >= iou_threshold and score >= threshold:
                        true_positives += 1
                    elif score >= threshold:
                        false_positives += 1

            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / total_positives if total_positives > 0 else 0

            precisions.append(precision)
            recalls.append(recall)

        if not precisions or not recalls:
            return [0], [0]

        return precisions, recalls

    predictions_v1 = [result.get("mask_v1", None) for result in results]
    predictions_v2 = [result.get("mask_v2", None) for result in results]
    predictions_v3 = [result.get("mask_v3", None) for result in results]
    annotations = [result.get("annotation_mask", None) for result in results]

    scores_v1 = [0.9] * len(results)
    scores_v2 = [0.9] * len(results)
    scores_v3 = [0.9] * len(results)

    map_values = {"mAP_v1": {}, "mAP_v2": {}, "mAP_v3": {}}

    for iou_threshold in iou_thresholds:
        precisions_v1, recalls_v1 = compute_precision_recall(predictions_v1, annotations, scores_v1, iou_threshold)
        precisions_v2, recalls_v2 = compute_precision_recall(predictions_v2, annotations, scores_v2, iou_threshold)
        precisions_v3, recalls_v3 = compute_precision_recall(predictions_v3, annotations, scores_v3, iou_threshold)

        recalls_v1, precisions_v1 = zip(*sorted(zip(recalls_v1, precisions_v1)))
        recalls_v2, precisions_v2 = zip(*sorted(zip(recalls_v2, precisions_v2)))
        recalls_v3, precisions_v3 = zip(*sorted(zip(recalls_v3, precisions_v3)))

        map_v1 = auc(recalls_v1, precisions_v1)
        map_v2 = auc(recalls_v2, precisions_v2)
        map_v3 = auc(recalls_v3, precisions_v3)

        map_values["mAP_v1"][f"mAP@{int(iou_threshold * 100)}"] = map_v1
        map_values["mAP_v2"][f"mAP@{int(iou_threshold * 100)}"] = map_v2
        map_values["mAP_v3"][f"mAP@{int(iou_threshold * 100)}"] = map_v3

    return map_values

## Visualização das predições

In [ ]:
results, all_labels = generate_predictions(all_images, mso_v1, mso_v2, mso_v3)
plot_image_comparisons(results)

## Avaliação das métricas

In [ ]:
evaluate_predictions(results, all_labels)
display_statistics(results)

map_results = calculate_map(results, iou_thresholds=[0.5, 0.9])
print("\nmAP Results:")
for threshold, value in map_results["mAP_v1"].items():
    print(f"MSO_V1 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v2"].items():
    print(f"MSO_V2 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v3"].items():
    print(f"MSO_V3 - {threshold}: {value:.4f}")

In [ ]:
!jupyter nbconvert --to html Pred_Lote45_ModelsSegm.ipynb